In [11]:
import pandas as pd

df=pd.read_csv("C:\\Users\\ASUS\\Documents\\Springboard\\cleaned HR_Final_Preprocessed.csv")
print("shape:",df.shape)
df.head()

print(df.isnull().sum())
print("missing Elemenets",df.isnull().sum().sum())


shape: (1470, 39)
Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole    

In [12]:
textcol=df.select_dtypes(include='object').columns.tolist()
print(textcol)

['Attrition', 'BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'Over18', 'OverTime', 'Age_Group', 'Experience_Level', 'Income_Category', 'Tenure_Group']


C:\Users\ASUS\AppData\Local\Temp\ipykernel_5752\3281483449.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  textcol=df.select_dtypes(include='object').columns.tolist()


In [13]:
df['Attrition']=df['Attrition'].map({'Yes':1,'No':0})

print(df['Attrition'].value_counts())

Attrition
0    1233
1     237
Name: count, dtype: int64


In [14]:
df['OverTime'] = df['OverTime'].map({'Yes': 1, 'No': 0})
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['Over18'] = df['Over18'].map({'Y': 1})

In [15]:
multi_cat_cols = ['BusinessTravel', 'Department', 'EducationField', 'JobRole',
                   'MaritalStatus', 'Age_Group', 'Experience_Level',
                   'Income_Category', 'Tenure_Group']

df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)
print(df.shape)


(1470, 58)


In [16]:
num_cols=df.select_dtypes(include='bool').columns
df[num_cols]=df[num_cols].astype(int)

In [17]:
remaining_text = df.select_dtypes(include='object').columns.tolist()

if remaining_text:
    print("Still text:", remaining_text)
else:
    print("All numerical rows")

All numerical rows


In [45]:
df.to_excel("C:\\Users\\ASUS\\Documents\\Springboard\\encoded_data.xlsx",index=False)


In [18]:
x=df.drop('Attrition',axis=1)
y=df['Attrition']

print(x.shape)
print(y.value_counts())

(1470, 57)
Attrition
0    1233
1     237
Name: count, dtype: int64


In [26]:
#testing and training data

from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

print("Train size:", x_train.shape)
print("Test size:", x_test.shape)

Train size: (1176, 57)
Test size: (294, 57)


In [27]:
#logistic regression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(x_train, y_train)

print("Model trained!")


Model trained!


c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [28]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(x_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))
print()
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.6462585034013606

              precision    recall  f1-score   support

           0       0.94      0.62      0.75       247
           1       0.29      0.81      0.42        47

    accuracy                           0.65       294
   macro avg       0.61      0.71      0.58       294
weighted avg       0.84      0.65      0.69       294


[[152  95]
 [  9  38]]


c:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [29]:
#find trainers needs more development
df['Skill_Gap_Flag'] = ((df['TrainingTimesLastYear'] <= 1) & (df['PerformanceRating'] <= 3)).astype(int)

print(df['Skill_Gap_Flag'].value_counts())

Skill_Gap_Flag
0    1364
1     106
Name: count, dtype: int64


In [30]:
df['Promotion'] = ((df['YearsSinceLastPromotion'] >= 3) & (df['PerformanceRating'] >= 3) & (df['JobLevel'] < 5)).astype(int)

print(df['Promotion'].value_counts())

Promotion
0    1130
1     340
Name: count, dtype: int64


In [31]:
#Workforce health score\
health_cols = ['EnvironmentSatisfaction', 'JobSatisfaction', 'RelationshipSatisfaction', 'WorkLifeBalance']

df['Health_Score_Raw'] = df[health_cols].mean(axis=1)

df['Health_Score'] = ((df['Health_Score_Raw'] - df['Health_Score_Raw'].min()) /
                       (df['Health_Score_Raw'].max() - df['Health_Score_Raw'].min())) * 100

print(df[['Health_Score_Raw', 'Health_Score']].describe())


       Health_Score_Raw  Health_Score
count       1470.000000   1470.000000
mean           2.730952     57.698413
std            0.505815     16.860495
min            1.000000      0.000000
25%            2.500000     50.000000
50%            2.750000     58.333333
75%            3.000000     66.666667
max            4.000000    100.000000


In [33]:
df.to_excel("C:\\Users\\ASUS\\Documents\\Springboard\\encoded_data.xlsx",index=False)
